# Phase 4: multi-step agent, recovery paths, one round of expert iteration (Colab T4)
Stages (each is skip-if-done, so a disconnect loses nothing; each prints a time estimate; target about 60 min on a T4):
1. fine-tuned Qwen agent **without** recovery on the 174 test episodes
2. fine-tuned Qwen agent **with** recovery
3. expert-iteration rollouts on training episodes (sampling, temperature 0.8, recovery on, the simulator is the reward)
4. build the RL dataset from simulator-verified successes, mixed with the Phase 2 train set
5. continue LoRA fine-tuning from the Phase 3 adapter (1 epoch, lr 1e-4)
6. single-step eval of the new model, then the agent with recovery on the test episodes
7. results tables + a results zip (no weights)

**Needs:** the Phase 3 adapter on Drive at `runs/phase3/qwen25_05b_lora/adapter` (written by notebook 01), and your latest code **pushed to GitHub**.
**Data** comes from the git clone; **runs** are written to Drive under `runs/phase4/`; the last cell writes `runs/phase4_results.zip` (download it from drive.google.com, then `python scripts/import_results_zip.py <zip>` on the Mac).

**Kernel:** in VS Code choose **Select Kernel -> Colab -> New Colab Server -> GPU -> T4**. If Colab disconnects: reconnect and run all cells again.

In [ ]:
# ===== CONFIG: change settings here only =====
CONFIG = dict(
    base_model="Qwen/Qwen2.5-0.5B-Instruct",
    phase3_run="qwen25_05b_lora",      # the adapter to start from (runs/phase3/<this>/adapter)
    rl_run="qwen25_05b_rl",            # new run (runs/phase4/<this>)
    max_steps=8,
    # expert iteration
    n_train_episodes=704,              # seeded subset of the 1408 train episodes (all = 1408; time scales linearly)
    samples_per_episode=2,             # k rollouts per episode
    temperature=0.8,
    chunk_episodes=176,                # rollouts are saved in chunks of this many episodes
    rl_learning_rate=1e-4, rl_epochs=1,
    eval_batch_size=64,                # prompts per generate() call
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
REPO = "/content/spice-verified-analog-finetuning"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Keerthanareddy17/spice-verified-analog-finetuning.git", REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# torch comes preinstalled on Colab: do not reinstall it. This notebook also needs ngspice (the agent simulates every move).
# Colab ships an old torchao (0.10.0) that we never use; peft 0.21.2 aborts LoRA creation when it finds torchao < 0.16.
!pip -q uninstall -y torchao
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps
!apt-get -qq install -y ngspice > /dev/null
!ngspice --version | head -3

In [ ]:
# Fail early, before any model loads: print versions and run peft's optional-package checks (the torchao check is what broke before).
import importlib, importlib.metadata as md, importlib.util
importlib.invalidate_caches()
for pkg in ["torch", "transformers", "trl", "peft", "accelerate", "datasets", "torchao", "bitsandbytes"]:
    try:
        print(f"{pkg:14s}", md.version(pkg))
    except md.PackageNotFoundError:
        print(f"{pkg:14s}", "not installed")
import peft.import_utils as piu
from peft import get_peft_model, LoraConfig   # noqa: F401
bad = []
for name in sorted(n for n in dir(piu) if n.startswith("is_") and n.endswith("_available")):
    try:
        getattr(piu, name)()
    except ImportError as e:
        bad.append(f"{name}: {e}")
    except TypeError:
        pass   # needs arguments, not an environment check
assert not bad, "peft found an incompatible optional package:\n" + "\n".join(bad)
print("peft optional-package checks OK")

In [ ]:
import sys
sys.path.insert(0, REPO + "/src")
import importlib, transformers, trl, peft, torch
from analogft import paths
from analogft.drive_check import verify_manifest
print("colab:", paths.IS_COLAB, "| data:", paths.DATA_DIR, "| runs:", paths.RUNS_DIR)
print("transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__, "| torch", torch.__version__)
ok, problems = verify_manifest(paths.DATA_DIR)
assert ok, "data/ in the repo clone does not match data/manifest.json (did you push the data commit? run git pull): " + "; ".join(problems[:5])
print("data manifest OK (data read from the git clone:", paths.DATA_DIR, ")")

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("NO GPU: in VS Code use Select Kernel -> Colab -> New Colab Server -> GPU -> T4, then run all cells again.")
print("GPU:", torch.cuda.get_device_name(0), "| memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# The Colab ngspice must reproduce the Mac's answer for the nominal design (results/reference_nominal.json).
import json, os
from analogft.circuit import nominal_design, simulate
ref = json.load(open(paths.REPO_ROOT / "results" / "reference_nominal.json"))
r = simulate(nominal_design(), use_cache=False)
for k, v in ref["metrics"].items():
    got = r.metrics[k]
    assert abs(got - v) <= 1e-3 * abs(v) + 1e-12, f"ngspice disagrees with the Mac on {k}: {got} vs {v}"
print("ngspice matches the Mac reference:", {k: round(r.metrics[k], 4) for k in ("dc_gain_db", "pm_deg")}, "| cpus:", os.cpu_count())

In [ ]:
import time
from pathlib import Path
from analogft.agent import SimPool
from analogft.agent_eval import fmt_agent, load_episodes, run_and_save, seeded_subset
from analogft.dataset import read_jsonl
from analogft.expert_iteration import build_rl_dataset
from analogft.finetune import FTConfig, evaluate_model, train
from analogft.phase4_run import load_rollouts, run_rollouts, write_rollouts_summary
from analogft.policies import LLMPolicy
from analogft.predict import HFBackend

P4 = paths.RUNS_DIR / "phase4"
ADAPTER3 = paths.RUNS_DIR / "phase3" / CONFIG["phase3_run"] / "adapter"
assert (ADAPTER3 / "adapter_config.json").exists(), f"Phase 3 adapter not found at {ADAPTER3}: run notebook 01 first"
test_eps, train_eps = load_episodes("test"), load_episodes("train")
sim = SimPool(workers=max(2, os.cpu_count() or 2))
t_sim = time.time(); sim.run([nominal_design()] * 1); print("simulation pool ready;", len(test_eps), "test episodes,", len(train_eps), "train episodes")
backend = HFBackend(CONFIG["base_model"], ADAPTER3, batch_size=CONFIG["eval_batch_size"])   # greedy by default
policy = LLMPolicy(backend, name="finetuned_qwen")

## 1. Fine-tuned agent WITHOUT recovery (174 test episodes)

In [ ]:
m_ft0 = run_and_save(P4 / "agents" / "ft_norecovery", "ft_norecovery", policy, test_eps, recovery=False, sim=sim, skip_if_done=True, max_steps=CONFIG["max_steps"])
print(fmt_agent(m_ft0))

## 2. Fine-tuned agent WITH recovery

In [ ]:
m_ft1 = run_and_save(P4 / "agents" / "ft_recovery", "ft_recovery", policy, test_eps, recovery=True, sim=sim, skip_if_done=True, max_steps=CONFIG["max_steps"])
print(fmt_agent(m_ft1))

## 3. Expert-iteration rollouts on TRAIN episodes (sampling; chunked and resumable; prints a time estimate)

In [ ]:
sub_train = seeded_subset(train_eps, CONFIG["n_train_episodes"], seed=0)
print(f"{len(sub_train)} train episodes x {CONFIG['samples_per_episode']} samples at temperature {CONFIG['temperature']}")
backend.temperature = CONFIG["temperature"]
run_rollouts(policy, sub_train, P4 / "rollouts", sim, samples=CONFIG["samples_per_episode"], chunk_size=CONFIG["chunk_episodes"])
backend.temperature = 0.0
roll_steps, roll_eps = load_rollouts(P4 / "rollouts")
print(write_rollouts_summary(P4 / "rollouts", roll_eps, P4 / "rollouts_summary.json", CONFIG["samples_per_episode"], CONFIG["temperature"]))

## 4. RL dataset: only simulator-verified successes, improving steps the model chose itself (no rules fallback), no duplicates; mixed with the Phase 2 train set

In [ ]:
(P4 / "rl_dataset").mkdir(parents=True, exist_ok=True)
phase2_train = read_jsonl(paths.DATA_DIR / "phase2" / "train.jsonl")
mixed, rl_stats = build_rl_dataset(phase2_train, roll_steps, roll_eps, P4 / "rl_dataset" / "rl_train.jsonl", seed=0)
import json
print(json.dumps(rl_stats, indent=1))
assert rl_stats["new_examples"] > 0, "no new examples: nothing to learn from (see the stats above)"

## 5. Continue fine-tuning from the Phase 3 adapter (1 epoch, lr 1e-4, same LoRA config and fp16 handling)

In [ ]:
import gc, torch
del backend, policy; gc.collect(); torch.cuda.empty_cache()
cfg_rl = FTConfig(base_model=CONFIG["base_model"], run_name=CONFIG["rl_run"], phase="phase4", init_adapter=str(ADAPTER3),
                  train_file=str(P4 / "rl_dataset" / "rl_train.jsonl"), epochs=CONFIG["rl_epochs"],
                  learning_rate=CONFIG["rl_learning_rate"], per_device_batch=16, grad_accum=1)
if (cfg_rl.run_dir / "adapter" / "adapter_config.json").exists():
    print("RL adapter already trained:", cfg_rl.run_dir / "adapter")
else:
    rl_summary = train(cfg_rl)

## 6. Single-step eval of the new model (compare with Phase 3: no regression?), then the agent with recovery

In [ ]:
rl_res = evaluate_model(cfg_rl, "finetuned", cfg_rl.run_dir / "adapter", skip_if_done=True)

In [ ]:
backend_rl = HFBackend(CONFIG["base_model"], cfg_rl.run_dir / "adapter", batch_size=CONFIG["eval_batch_size"])
policy_rl = LLMPolicy(backend_rl, name="rl_qwen")
m_rl = run_and_save(P4 / "agents" / "rl_recovery", "rl_recovery", policy_rl, test_eps, recovery=True, sim=sim, skip_if_done=True, max_steps=CONFIG["max_steps"])
print(fmt_agent(m_rl))
sim.close()

## 7. Results tables (Mac agents come from the committed results/phase4) and the results zip

In [ ]:
!python scripts/make_phase4_table.py

In [ ]:
from analogft.resultszip import make_phase4_zip
(P4 / "figures").mkdir(parents=True, exist_ok=True)
import shutil
shutil.copy(paths.REPO_ROOT / "results" / "figures" / "phase4_success_vs_budget.png", P4 / "figures" / "phase4_success_vs_budget.png")
z = make_phase4_zip(P4)
print(f"wrote {z}  ({z.stat().st_size / 1e6:.2f} MB)")
print(f"On drive.google.com open:  My Drive / {paths.PROJECT_NAME} / runs / {z.name}")
print(f"then download it and on the Mac run:  python scripts/import_results_zip.py ~/Downloads/{z.name}")

In [ ]:
from IPython.display import Image, display
display(Image(str(paths.REPO_ROOT / "results" / "figures" / "phase4_success_vs_budget.png")))